# SISR Model Overview

This notebook introduces all model families in the framework, explains their design philosophies, and shows how to instantiate each model.

## Taxonomy

| Family | Models | Key Idea | Objective |
|--------|--------|----------|-----------|
| Interpolation | Nearest, Bilinear, Bicubic | Fixed kernels | Baseline |
| Shallow CNN | SRCNN | 3-layer CNN | PSNR |
| Deep Residual CNN | VDSR, EDSR, RCAN | Skip + depth | PSNR |
| Lightweight | ESPCN, IMDN | LR-space + pixel shuffle | PSNR/Edge |
| GAN-based | SRGAN, ESRGAN, Real-ESRGAN | Adversarial | Perceptual |
| Transformer | SwinIR, HAT | Self-attention | PSNR/Perceptual |
| Diffusion | SR3 | Iterative denoising | Generative |

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import models  # registers all models
from models.registry import list_models, build_model
from omegaconf import OmegaConf

print('Registered models:', list_models())

## Instantiate All Models

In [ ]:
from models.interpolation.classical import BicubicSR
from models.cnn.srcnn import SRCNN
from models.cnn.vdsr import VDSR
from models.cnn.edsr import EDSR
from models.cnn.rcan import RCAN
from models.cnn.espcn import ESPCN
from models.gan.srgan import SRGAN
from models.gan.esrgan import ESRGAN
from models.transformer.swinir import SwinIR
from models.transformer.hat import HAT
from models.lightweight.imdn import IMDN

models_to_show = {
    'Bicubic':  BicubicSR(scale=4),
    'SRCNN':    SRCNN(scale=4, in_channels=1, out_channels=1),
    'VDSR':     VDSR(scale=4, in_channels=1, out_channels=1),
    'EDSR':     EDSR(scale=4),
    'RCAN':     RCAN(scale=4, num_groups=3, num_rcab_per_group=4),  # small for demo
    'ESPCN':    ESPCN(scale=4),
    'SRGAN':    SRGAN(scale=4),
    'ESRGAN':   ESRGAN(scale=4, num_rrdb=5),  # small for demo
    'SwinIR':   SwinIR(scale=4, embed_dim=32, depths=[2,2], num_heads=[2,2]),  # small
    'HAT':      HAT(scale=4, embed_dim=48, depths=[2,2], num_heads=[2,2], window_size=8),
    'IMDN':     IMDN(scale=4),
}

print(f'{'Model':<15} {'Parameters':>15}')
print('-' * 32)
for name, m in models_to_show.items():
    params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:<15} {params:>15,}')

## Forward Pass Shapes

In [ ]:
dummy_rgb = torch.rand(1, 3, 32, 32)
dummy_y   = torch.rand(1, 1, 32, 32)

print(f'Input LR shape: {dummy_rgb.shape}')
print()

for name, m in models_to_show.items():
    m.eval()
    with torch.no_grad():
        inp = dummy_y if m.in_channels == 1 else dummy_rgb
        try:
            out = m(inp)
            print(f'{name:<15}  →  {tuple(out.shape)}')
        except Exception as e:
            print(f'{name:<15}  ✗  {e}')

## Architecture Diagrams

### SRCNN
```
LR → Bicubic(×4) → Conv(9×9, 64) → ReLU → Conv(1×1, 32) → ReLU → Conv(5×5, 1) → SR
```

### EDSR
```
LR → Conv → [ResBlock × 16] → Conv → PixelShuffle(×4) → Conv → SR
```

### RCAN
```
LR → Conv → [ResGroup × 10 → [RCAB × 20]] → Conv → PixelShuffle → Conv → SR
         RCAB = Conv-ReLU-Conv-ChannelAttn
```

### ESRGAN Generator (RRDB)
```
LR → Conv → [RRDB × 23] → Conv → PixelShuffle → Conv-LReLU-Conv → SR
     RRDB = 3 × DenseBlock (densely connected) with residual scaling
```

### SwinIR
```
LR → Conv → [RSTB × N] → LayerNorm → Conv → PixelShuffle → Conv → SR
     RSTB = [SwinTransformerLayer × depth] + Conv + residual
```

### HAT
```
LR → Conv → [HATBlock × N] → LayerNorm → Conv → PixelShuffle → Conv → SR
     HATBlock = [HAB × depth] + OCAB + Conv + residual
     HAB = WindowMSA + ChannelAttention + MLP
     OCAB = OverlappingCrossAttention
```

## Model Selection Guide

| Goal | Recommended Model | Why |
|------|-------------------|-----|
| Maximum PSNR (research) | HAT, RCAN | State-of-the-art benchmark scores |
| Strong PSNR (practical) | EDSR, SwinIR | Good balance of quality vs. compute |
| Realistic textures | ESRGAN, Real-ESRGAN | GAN hallucination of high-freq detail |
| Real-world blind SR | Real-ESRGAN | Trained with synthetic degradation |
| Lightweight/mobile | IMDN, ESPCN | <1M params, real-time capable |
| Generative quality | SR3 | Diffusion prior, best for diversity |